In [1]:
import os
# os.environ["MUJOCO_GL"] = "glfw"

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot training
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.configs.default import WandBConfig
from lerobot.utils.utils import init_logging
from lerobot.scripts import train

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint
import sys

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
PUSH_TO_HUB       = False
WANDB             = True
REPO_NAME         = 'so101_car_pick_and_place'
EXPERIMENT_NAME   = 'extended_v0'
RESUME_CHECKPOINT = None
DATASET_PATH      = DATASETS_DIR / REPO_NAME
DATASET_ID        = f"{HF_NAME}/{REPO_NAME}"
POLICY_ID         = f"{HF_NAME}/act-{REPO_NAME}-{EXPERIMENT_NAME}"

In [4]:
if PUSH_TO_HUB:
    os.system(f"hf auth login --token {os.getenv('HUGGINGFACE_TOKEN')}")
if WANDB:
    wandb.login(key = os.getenv('WANDB_API_KEY'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/jonathan/.netrc
wandb: Currently logged in as: jonathm126 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"\nRobot type: {ds_meta.robot_type}")
print(f"\nkeys to access images from cameras: {ds_meta.camera_keys}")
print("\nFeatures:")
pprint(ds_meta.features.keys(), width = 40 )

Total number of episodes: 49
Average number of frames per episode: 769.898
Frames per second used during data collection: 30

Robot type: so101_follower

keys to access images from cameras: ['observation.images.wrist_cam', 'observation.images.top_cam']

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'observation.environment_state'])


In [6]:
# select image transforms
transforms_cfg = ImageTransformsConfig(
    enable             = False, # note about augmentations
    max_num_transforms = 3,
    random_order       = True,
)

# build dataset
dataset_cfg = DatasetConfig(
    repo_id            = DATASET_ID,
    root               = DATASET_PATH.__str__(),   # dataset location
    image_transforms   = transforms_cfg,           # configure image transformations
    use_imagenet_stats = True,                     # normalize image using imagenet weights
    video_backend      = 'torchcodec',
    streaming          = False,
)

Policy Config

In [7]:
input_features = {
    "observation.images.wrist_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.images.top_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.state": {
        "type" : "STATE",
        "shape": [6]
    },
    "observation.environment_state": {
        "type" : "STATE",
        "shape": [3]
    },
}

output_features = {
    "action": {
        "type" : "ACTION",
        "shape": [6]
    }
}

In [8]:
policy_config = ACTConfig(
    n_obs_steps                 = 1,                                  # policy sample freq - should be 1
    chunk_size                  = 100,                                # chunk size outut from the moodel
    n_action_steps              = 100,                                # use only 15 at inference (0.5 sec)
    input_features              = input_features,
    output_features             = output_features,
    device                      = device,
    use_amp                     = False,                              # note about amp
    push_to_hub                 = PUSH_TO_HUB,
    repo_id                     = POLICY_ID,
    vision_backbone             = 'resnet18',
    pretrained_backbone_weights = 'ResNet18_Weights.IMAGENET1K_V1',
    dim_model                   = 512,
    use_vae                     = True,                               # should you use a encoder to learn the style variable
    latent_dim                  = 32,                                 # of the latent Z encoder
    n_vae_encoder_layers        = 4,                                  # of the latent Z encoder
    temporal_ensemble_coeff     = None,                               # temporal averaging coefficient, nominal is 0.01
    kl_weight                   = 1,                                  # defualt is 10
    optimizer_lr                = 1e-5                                # default is 1e-5
    )

WandB logging:

In [9]:
wandb_config = WandBConfig(
    enable           = WANDB,
    disable_artifact = True,
    project          = 'ACT-Real-Sim-Real',
    entity           = 'jonathm126-personal',
    mode             = 'online',
    run_id           = f'act-{REPO_NAME}-{EXPERIMENT_NAME}-train'  # careful - this is a unique ids
)
print(wandb_config.run_id)


act-so101_car_pick_and_place-extended_v0-train


Integration:

In [10]:
train_cfg = TrainPipelineConfig(
    dataset                    = dataset_cfg,
    # env                        = env_config,
    policy                     = policy_config,                                                     # policy config
    output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
    job_name                   = POLICY_ID + '-train',                                              # name
    num_workers                = 4,
    batch_size                 = 12,                                                                 # check for stabillity
    steps                      = 60000,                                                             # check for number of epochs - we want 15-20 epochs
    eval_freq                  = 20000,
    log_freq                   = 200,
    save_checkpoint            = True,
    save_freq                  = 20000,
    use_policy_training_preset = True,                                                              # for scheduler and optimizer
    eval                       = None,
    wandb                      = wandb_config,
)

Train:

In [12]:
if RESUME_CHECKPOINT is None:
    init_logging()
    train.train(train_cfg)
else:
    assert RESUME_CHECKPOINT is not None
    pth = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / resume_checkpoint / 'pretrained_model' / 'train_config.json'
    assert pth.exists()
    sys.argv = [
        "train",
        f"--config_path={pth}",
        "--resume=true",
    ]
    init_logging()
    train.train()

INFO 2025-10-12 19:54:12 ts/train.py:148 {'batch_size': 12,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': True,
                                  'tfs': {'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                                                         'type': 'ColorJitter',
                                                         'weight': 1.0},
                                          'contrast': {'kwargs': {'contrast': [0.8,
                                                                               1.2]},
                                                       'type': 'ColorJitter',
                                                       'weight': 1.0},
                                          'hue': {'kwargs': {'hue': [-0.05,
                 

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
INFO 2025-10-12 19:54:12 db_utils.py:103 Track this run --> https://wandb.ai/jonathm126-personal/ACT-Real-Sim-Real/runs/act-so101_car_pick_and_place-extended_v0-train
INFO 2025-10-12 19:54:12 ts/train.py:164 Creating dataset


Logs will be synced with wandb.


INFO 2025-10-12 19:54:13 ts/train.py:175 Creating policy
INFO 2025-10-12 19:54:14 ts/train.py:194 Creating optimizer and scheduler
INFO 2025-10-12 19:54:14 ts/train.py:206 Output dir: /home/jonathan/Github/IDC_Project-Sim_Real_Sim/policies/act/so101_car_pick_and_place/extended_v0
INFO 2025-10-12 19:54:14 ts/train.py:209 cfg.steps=60000 (60K)
INFO 2025-10-12 19:54:14 ts/train.py:210 dataset.num_frames=72648 (73K)
INFO 2025-10-12 19:54:14 ts/train.py:211 dataset.num_episodes=49
INFO 2025-10-12 19:54:14 ts/train.py:212 num_learnable_params=51599750 (52M)
INFO 2025-10-12 19:54:14 ts/train.py:213 num_total_params=51599750 (52M)
INFO 2025-10-12 19:54:14 ts/train.py:254 Start offline training on a fixed dataset
INFO 2025-10-12 19:55:26 ts/train.py:281 step:200 smpl:2K ep:2 epch:0.03 loss:1.030 grdn:26.543 lr:1.0e-05 updt_s:0.357 data_s:0.005
INFO 2025-10-12 19:56:36 ts/train.py:281 step:400 smpl:5K ep:3 epch:0.07 loss:0.541 grdn:19.845 lr:1.0e-05 updt_s:0.346 data_s:0.001
INFO 2025-10-12 19:5